In [1]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.io as pio
from src.score import Score
import os 
import toml
from tabulate import tabulate
from tqdm import tqdm
import re 

# Import
from pandarallel import pandarallel

# Initialization
pandarallel.initialize(progress_bar=True, nb_workers=8)



current_dir = os.getcwd()
con = sqlite3.connect("data/main_database.db")
# pd.set_option('display.float_format', lambda x: '%.3f' % x)
pd.set_option('display.max_columns', None)
connew = sqlite3.connect("data/text/wiki/wikipedia2.sqlite")



INFO: Pandarallel will run on 8 workers.
INFO: Pandarallel will use standard multiprocessing data transfer (pipe) to transfer data between the main process and workers.


In [5]:
df = pd.read_sql_query('select * from valcount', connew)
df.head(20)

,index,clean_article
0,the,67170106
1,of,37460583
2,and,27822269
3,in,25543874
4,a,21891867
5,to,19185534
6,s,11093262
7,on,9718075
8,for,8231982
9,from,7982007


In [6]:
export = [1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000, 1000000, 2000000, 5000000]    

for value in tqdm(export):
    if value > len(df):
        value = len(df)
    word_list = df.iloc[:value, 0].tolist()
    print(len(word_list))

    with open(f'data/text/wikipedia_{value}.txt', 'w') as file:
        file.write('\n'.join(word_list))


 83%|████████▎ | 10/12 [00:00<00:00, 95.10it/s]

1000
2000
5000
10000
20000
50000
100000
200000
500000
1000000
2000000


100%|██████████| 12/12 [00:00<00:00, 23.22it/s]

5000000


In [ ]:
all_words = pd.read_sql_query('select distinct * from pages', connew)['text'].to_list()
all_words[:10]

In [ ]:
from unidecode import unidecode
unidecode('oké')

In [ ]:
directory = 'data/text/'   
for file in tqdm(os.listdir(directory)):
    file = os.path.join(directory, file)
    if '.txt' in file: 
        with open(file, 'w+') as f: 
            f.write(re.sub(r'[^a-z]+', ' ', unidecode(f.read())))

In [ ]:
# all_words = [re.sub(r'[^a-z]+', ' ', unidecode(article.lower())) for article in tqdm(all_words)]

In [ ]:
# Create and clean dataframe
df = pd.DataFrame({'word': all_words})

In [ ]:
df = df.drop_duplicates()

In [ ]:
df['clean_article'] = df['word'].parallel_apply(lambda article: re.sub(r'[^a-z]+', ' ', unidecode(article.lower())))
df.to_sql('clean_article', connew, if_exists='replace', index=False)

In [2]:
df = pd.read_sql_query('select * from clean_article', connew)

In [ ]:
df.shape

In [3]:
df = pd.read_sql_query('select * from clean_article', connew)

lst = df.loc[:len(df) // 1, 'clean_article'].to_list()
lst = ' '.join(lst).split(' ')
print(len(lst))




# df.groupby(args).apply(func)
for i in tqdm(range(37, 1000)):
    valcount = df.loc[len(df) // 1000 * i:len(df) // 1000 * (i+1), 'clean_article'].str.split(' ').explode().value_counts().to_frame().reset_index()
    valcount.to_sql('valcount', connew, if_exists='append', index=False)

1325619152


In [2]:
connew.execute('drop table valcount')

OperationalError: no such table: valcount

In [1]:



# df.groupby(args).apply(func)
for i in tqdm(range(37, 1000)):
    valcount = df.loc[len(df) // 1000 * i:len(df) // 1000 * (i+1), 'clean_article'].str.split(' ').explode().value_counts().to_frame().reset_index()
    valcount.to_sql('valcount', connew, if_exists='append', index=False)

NameError: name 'tqdm' is not defined

In [8]:
df = pd.read_sql_query('select * from valcount', connew)
df.groupby('index').sum().sort_values('clean_article', ascending=False).to_sql('valcount', connew, if_exists='replace', index=True)

KeyboardInterrupt: 

In [6]:
df.groupby('index', as_index=False).sum().to_sql('valcount', connew, if_exists='replace', index=True)

4291366

In [ ]:
word_freq = df.groupby('words').size().reset_index(name='frequency')


In [ ]:
word_freq = {}  # Dictionary to store word frequencies

for word in tqdm(lst[:len(lst) // 10000000]):
    if word in word_freq:
        word_freq[word] += 1
    else:
        word_freq[word] = 1

In [ ]:
os.system('say "done"')

In [ ]:
df['clean_article'].str.split(' ')#.explode().value_counts()

In [ ]:
df['clean_article'].str.split(' ').explode().value_counts().to_frame().reset_index().rename({'index': 'word', 'word': 'count'}, axis=1).to_sql('wikipedia_word_frequency', connew, index=False, if_exists='replace')

In [ ]:
pd.read_sql_query('select * from wikipedia_word_frequency', connew).iloc[:10]

In [ ]:
df = pd.read_sql_query('select word from wikipedia_word_frequency', connew).iloc[0]
df

In [ ]:
export = [1000, 2000, 5000, 10000, 20000, 50000, 100000, 200000, 500000]    

for value in tqdm(export):
    word_list = df.iloc[:value, 0].tolist()
    print(len(word_list))

    with open(f'data/text/wikipedia_{value}.txt', 'w') as file:
        file.write('\n'.join(word_list))


In [ ]:
from unidecode import unidecode

def remove_accents(word):
    return unidecode(word)

# Example usage
# words = ["adhyatmik", "subalterns", "cashers", "hyökkäys", "ґghe", "girne", "liviana", "pereq",
#          "reverol", "alcáçovas", "egolf", "vtethelemaorganizations", "careerfields", "persiaran",
#          "isight", "kitáb", "neysani", "plaviuk", "voxup", "archiverules", "makami", "echiniscus",
#          "kajana", "ähtär"]

# modified_words = [remove_accents(word) for word in words]

# Print modified words
# for word in modified_words:
#     print(word)
remove_accents(' adhyatmik subalterns cashers hyökkäys ґghe girne liviana pereq reverol alcáçovas egolf vtethelemaorganizations careerfields persiaran isight kitáb neysani plaviuk voxup archiverules makami echiniscus kajana ähtär')

In [ ]:
counter = {word: all_words.count(word) for word in tqdm(set(all_words))}